# Sentiment Analysis MBG (Tanpa SMOTE, Tanpa scikit-learn)

Notebook ini:
- Membaca dataset mentah **`datasetmbg.csv`** (pemisah kolom `|`)
- Membersihkan teks sederhana (rule-based)
- **Auto-label** sentimen pakai **Indo-RoBERTa** (`w11wo/indonesian-roberta-base-sentiment-classifier`)
- Menyimpan hasil label ke **`dataset_sentimen_berlabel.csv`**
- Melatih model baseline **TF-IDF + Multinomial Naive Bayes** **tanpa scikit-learn** (NumPy + SciPy)

> Catatan: pastikan file `datasetmbg.csv` ada di folder yang sama dengan notebook ini.


In [112]:
# 1) Install deps (kalau perlu)
# Jalankan cell ini jika environment kamu belum ada paket-paket berikut.
# (Di beberapa lomba, instalasi dilarang—kalau begitu, pastikan paketnya sudah tersedia.)

!pip -q install pandas numpy scipy tqdm transformers torch


In [36]:
# 2) Imports
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from transformers import pipeline
from scipy.sparse import csr_matrix
from IPython.display import clear_output, display


## 3) Load dataset mentah

Format yang didukung:
- `datasetmbg.csv` dengan separator `|`

Kolom yang didukung:
- `text` (disarankan) atau `clean_text`
- Jika ternyata sudah ada kolom `label`/`sentiment`, notebook akan **mengabaikannya** dan melakukan auto-label ulang (biar konsisten).


In [37]:
# 3) Load dataset mentah
PATH_IN = ('/PyAi/datasetmbgnolabel.csv')

df = pd.read_csv(PATH_IN, sep="|", encoding='latin1')
print("Kolom:", df.columns.tolist())
df.head()

Kolom: ['text']


,text
0,MBG bikin anak lebih fokus belajar karena peru...
1,"Program MBG bantu orang tua, apalagi yang peng..."
2,"Seneng lihat anak dapat menu sehat di sekolah,..."
3,MBG kalau konsisten bisa bantu turunkan stunti...
4,"Menu MBG di sekolahku lumayan variatif, anak j..."


In [38]:
# 4) Normalisasi nama kolom teks
if "text" in df.columns:
    df["clean_text"] = df["text"].astype(str)
elif "clean_text" in df.columns:
    df["clean_text"] = df["clean_text"].astype(str)
else:
    raise ValueError("datasetmbg.csv harus punya kolom 'text' atau 'clean_text'")

# Abaikan label kalau ada
for c in ["label", "sentiment", "target"]:
    if c in df.columns:
        print(f"Mengabaikan kolom label yang ada: {c}")

df = df[["clean_text"]].copy()
df["clean_text"] = df["clean_text"].astype(str).fillna("").str.strip()
df = df[df["clean_text"].str.len() > 0].copy()
                                                                                                                                               '3cd x
print("Jumlah baris:", len(df))
df.head()


Jumlah baris: 297


,clean_text
0,MBG bikin anak lebih fokus belajar karena peru...
1,"Program MBG bantu orang tua, apalagi yang peng..."
2,"Seneng lihat anak dapat menu sehat di sekolah,..."
3,MBG kalau konsisten bisa bantu turunkan stunti...
4,"Menu MBG di sekolahku lumayan variatif, anak j..."


## 4) Preprocessing teks (simple)

Ini cleaning ringan (boleh kamu modif):
- lowercase
- hapus URL, mention, hashtag symbol, angka
- hapus karakter non-huruf (kecuali spasi)
- rapihin spasi


In [39]:
import re
from tqdm.auto import tqdm
tqdm.pandas()

# (Opsional) Stemming Indo tanpa sklearn
USE_STEMMING = True
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _stemmer = StemmerFactory().create_stemmer()
except Exception:
    USE_STEMMING = False
    _stemmer = None

# Stopwords ID (ringkas tapi efektif). Bisa kamu tambah.
STOPWORDS_ID = {
    "yang","dan","di","ke","dari","untuk","pada","dengan","atau","itu","ini","aja","sih","nih",
    "gue","gw","aku","kamu","lu","loe","dia","mereka","kita","kami","anda","kak","bang",
    "nya","lah","deh","dong","kok","ya","yah","yahh","lahh","pun","juga","lagi","udah","sudah",
    "nggak","gak","ga","tak","tdk","tidak","bukan",
    "jadi","bisa","biar","supaya","agar","banget","bgt","sangat","amat",
    "tuh","tu","mah","kan","kayak","kaya","gini","gitu","sama","doang","cuma"
}

# Normalisasi slang/alay umum (boleh kamu expand)
SLANG_MAP = {
    "gk":"gak","ga":"gak","gak":"gak","ngga":"gak","nggak":"gak","tdk":"tidak",
    "bgt":"banget","bngt":"banget","bgtt":"banget",
    "yg":"yang","dgn":"dengan","utk":"untuk","dr":"dari","tp":"tapi","krn":"karena",
    "emg":"memang","aja":"saja","sm":"sama","km":"kamu","sy":"saya",
    "udh":"sudah","udah":"sudah","blm":"belum","skrg":"sekarang",
    "org":"orang","anak2":"anak anak","krg":"kurang","bkn":"bukan"
}

# Regex cepat
RE_URL = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
RE_MENTION = re.compile(r"@\w+")
RE_HASHTAG = re.compile(r"#(\w+)")
RE_HTML = re.compile(r"&amp;|&lt;|&gt;|&quot;|&#39;")
RE_NONALNUM = re.compile(r"[^0-9a-zA-Z\s]")
RE_MULTI_SPACE = re.compile(r"\s+")ds
RE_REPEAT_CHARS = re.compile(r"(.)\1{2,}")     # huruf berulang 3+ kali
RE_LAUGH = re.compile(r"\b(wk+|wkwk+|haha+|hehe+|hihi+|xixix+)\b", re.IGNORECASE)

def normalize_slang(token: str) -> str:
    return SLANG_MAP.get(token, token)

def reduce_elongation(word: str) -> str:
    # contoh: "baguuuuus" -> "baguus" (mengurangi jadi max 2)
    return RE_REPEAT_CHARS.sub(r"\1\1", word)

def basic_clean_competition(text: str) -> str:
    if text is None:
        return ""

    # lowercase
    text = str(text).lower()

    # unescape html sederhana
    text = RE_HTML.sub(" ", text)

    # hapus url, mention
    text = RE_URL.sub(" ", text)
    text = RE_MENTION.sub(" ", text)

    # hashtag: keep token-nya, buang '#'
    text = RE_HASHTAG.sub(r" \1 ", text)

    # ubah laughter jadi token khusus (biar konsisten)
    text = RE_LAUGH.sub(" LAUGH ", text)

    # normalisasi tanda baca berlebihan: !!!! -> !
    text = re.sub(r"([!?.,])\1{1,}", r"\1", text)

    # buang karakter aneh (sisain spasi)
    text = RE_NONALNUM.sub(" ", text)

    # normalisasi angka: bisa dibuang atau jadi token NUM
    text = re.sub(r"\b\d+\b", " NUM ", text)

    # rapihin spasi dulu
    text = RE_MULTI_SPACE.sub(" ", text).strip()

    # tokenisasi whitespace
    tokens = text.split()

    cleaned = []
    for tok in tokens:
        tok = normalize_slang(tok)
        tok = reduce_elongation(tok)

        # buang token terlalu pendek (kecuali "mbg" dll)
        if len(tok) <= 2 and tok not in {"mbg"}:
            continue

        # stopwords
        if tok in STOPWORDS_ID:
            continue

        cleaned.append(tok)

    text = " ".join(cleaned)

    # stemming (opsional)
    if USE_STEMMING and _stemmer is not None and text:
        text = _stemmer.stem(text)

    return text.strip()

# Apply (dengan progress bar)
df["clean_text"] = df["clean_text"].astype(str).progress_apply(basic_clean_competition)
df = df[df["clean_text"].str.len() > 0].copy()
df.head()

  0%|          | 0/297 [00:00<?, ?it/s]

,clean_text
0,mbg bikin anak lebih fokus belajar karena peru...
1,program mbg bantu orang tua apalagi penghasila...
2,seneng lihat anak dapat menu sehat sekolah jaj...
3,mbg kalau konsisten bantu turunkan stunting an...
4,menu mbg sekolahku lumayan variatif anak doyan...


## 5) Auto-label sentimen pakai Indo-RoBERTa

Output model biasanya `positive` / `negative` (kadang ada model lain yang punya `neutral`).
Di bawah ini:
- simpan `sentiment` (string)
- map ke `label` numerik: negative=0, positive=1
- kalau ada `neutral`, kita buang (atau kamu bisa map ke kelas ke-3)


In [40]:
# 5) Load pipeline (butuh internet saat pertama kali download model)
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
# 6) Inference dengan progress bar

PRINT_EVERY = 10   # setiap 10 teks, contoh diganti
ALT_SHOW = 6       # total contoh yg ditampilkan (mis. 6 = 3 pos + 3 neg)

results = []
texts = df["clean_text"].tolist()

# buffer utk bikin tampilannya selang-seling
buf_pos = []
buf_neg = []

pbar = tqdm(texts, desc="Analisis Sentimen")

for i, text in enumerate(pbar, start=1):
    out = sentiment_pipeline(text, truncation=True)[0]
    results.append(out)

    lbl = out["label"].lower().strip()
    sc  = float(out.get("score", np.nan))

    if lbl == "positive":
        buf_pos.append((sc, text))
    elif lbl == "negative":
        buf_neg.append((sc, text))
    # kalau ada label lain (mis. neutral), abaikan / bisa kamu handle sendiri

    # setiap N teks: hapus output lama + tampilkan contoh terbaru (bukan nambah ke bawah)
    if i % PRINT_EVERY == 0:
        clear_output(wait=True)     # <- ini yang bikin "replace"
        display(pbar)               # tampilkan progress bar lagi

        print(f"Sample (replace) @ {i}/{len(texts)} — selang-seling POS/NEG\n")

        take = max(1, ALT_SHOW // 2)
        pos_take = buf_pos[-take:]
        neg_take = buf_neg[-take:]

        shown = 0
        p = 0
        n = 0
        while shown < ALT_SHOW and (p < len(pos_take) or n < len(neg_take)):
            if p < len(pos_take) and shown < ALT_SHOW:
                scp, tp = pos_take[p]
                print(f"[POS] ({scp:.3f}) {tp[:200]}")
                p += 1
                shown += 1
            if n < len(neg_take) and shown < ALT_SHOW:
                scn, tn = neg_take[n]
                print(f"[NEG] ({scn:.3f}) {tn[:200]}")
                n += 1
                shown += 1

# simpan hasil ke df
df["sentiment"] = [r["label"].lower().strip() for r in results]
df["score"] = [float(r.get("score", np.nan)) for r in results]

df["sentiment"].value_counts()

Sample (replace) @ 290/297 — selang-seling POS/NEG

[POS] (0.990) stop gaya gayaan kalau gizinya jelas bagi nasi
[NEG] (0.986) jangan bilang bergizi buktinya mana kandungan gizinya dicek
[POS] (0.675) kadang menunya kebanyakan minyak gula sehat
[NEG] (0.998) kalau datangnya mepet pulang namanya asal asalan
[POS] (0.621) kalau makanan segar mending jangan dibagiin bahaya
[NEG] (0.998) mbg bikin pemborosan kalau banyak kebuang terus siapa mikirin


sentiment
negative    156
positive     82
neutral      59
Name: count, dtype: int64

In [42]:
# 7) Buat label numerik (binary). Drop neutral kalau muncul.
allowed = {"positive", "negative"}
df = df[df["sentiment"].isin(allowed)].copy()

label_map = {"negative": 0, "positive": 1}
df["label"] = df["sentiment"].map(label_map).astype(int)

print("Distribusi label:")
print(df["sentiment"].value_counts())
df.head()


Distribusi label:
sentiment
negative    156
positive     82
Name: count, dtype: int64


,clean_text,sentiment,score,label
0,mbg bikin anak lebih fokus belajar karena peru...,positive,0.590928,1
1,program mbg bantu orang tua apalagi penghasila...,positive,0.852039,1
2,seneng lihat anak dapat menu sehat sekolah jaj...,positive,0.555768,1
4,menu mbg sekolahku lumayan variatif anak doyan...,positive,0.997839,1
5,mbg bikin jam makan lebih tertib anak rewel,negative,0.487516,0


In [43]:
# 8) Simpan dataset berlabel (siap training)
OUT_LABELED = "dataset_sentimen_berlabel.csv"
df[["clean_text", "sentiment", "label", "score"]].to_csv(OUT_LABELED, index=False)
print(f"Selesai! File '{OUT_LABELED}' sudah siap dipakai.")


Selesai! File 'dataset_sentimen_berlabel.csv' sudah siap dipakai.


## 6) TF-IDF (tanpa scikit-learn)

Implementasi TF-IDF sederhana:
- Tokenisasi whitespace
- Build vocabulary dari train
- Hitung TF-IDF sparse matrix (CSR)


In [44]:
def train_test_split_np(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(X))
    rng.shuffle(idx)
    n_test = int(len(X) * test_size)
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X = df["clean_text"].to_numpy()
y = df["label"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split_np(X, y, test_size=0.2, seed=42)
len(X_train), len(X_test)


(191, 47)

In [45]:
def tokenize(text: str):
    return text.split()

def build_vocab(texts, min_df=2):
    # min_df: token muncul minimal berapa dokumen
    df_counts = {}
    for t in texts:
        seen = set(tokenize(t))
        for tok in seen:
            df_counts[tok] = df_counts.get(tok, 0) + 1

    # Filter tokens based on min_df first, then assign contiguous indices
    filtered_tokens = [tok for tok, cnt in df_counts.items() if cnt >= min_df]
    vocab = {tok: i for i, tok in enumerate(filtered_tokens)}
    return vocab, df_counts

vocab, df_counts = build_vocab(X_train, min_df=2)
V = len(vocab)
print("Vocab size:", V)

Vocab size: 245


In [46]:
def tfidf_matrix(texts, vocab, df_counts, n_docs_train):
    # Build CSR sparse TF-IDF
    rows = []
    cols = []
    data = []
    for i, t in enumerate(texts):
        toks = tokenize(t)
        if not toks:
            continue
        tf = {}
        for tok in toks:
            j = vocab.get(tok)
            if j is None:
                continue
            tf[j] = tf.get(j, 0) + 1
        if not tf:
            continue
        # compute tf-idf
        for j, cnt in tf.items():
            tok = None  # not needed
            # idf: log((N+1)/(df+1)) + 1
            # need df for this token: recover token by reverse? better precompute idf per index
            rows.append(i)
            cols.append(j)
            data.append(cnt)
    if len(data) == 0:
        return csr_matrix((len(texts), len(vocab)), dtype=np.float64)

    tf_mat = csr_matrix((np.array(data, dtype=np.float64),
                         (np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32))),
                        shape=(len(texts), len(vocab)))

    # L2-normalized TF part (optional). We'll use log-TF.
    tf_mat.data = 1.0 + np.log(tf_mat.data)

    # Precompute idf per column index using df_counts
    # Build array idf[V]
    idf = np.ones(len(vocab), dtype=np.float64)
    for tok, j in vocab.items():
        dfc = df_counts.get(tok, 0)
        idf[j] = np.log((n_docs_train + 1.0) / (dfc + 1.0)) + 1.0

    tf_mat = tf_mat.multiply(idf)

    # Normalize rows to unit length
    row_norm = np.sqrt(tf_mat.multiply(tf_mat).sum(axis=1)).A1
    row_norm[row_norm == 0] = 1.0
    tf_mat = tf_mat.multiply(1.0 / row_norm[:, None])
    return tf_mat

Xtr = tfidf_matrix(X_train, vocab, df_counts, n_docs_train=len(X_train))
Xte = tfidf_matrix(X_test, vocab, df_counts, n_docs_train=len(X_train))

Xtr.shape, Xte.shape


((191, 245), (47, 245))

## 7) Multinomial Naive Bayes (tanpa scikit-learn)

Kita pakai versi sederhana:
- hitung log prior
- hitung log likelihood dengan smoothing (alpha)


In [47]:
class MultinomialNB_NoSklearn:
    def __init__(self, alpha=1.0):
        self.alpha = float(alpha)
        self.class_log_prior_ = None
        self.feature_log_prob_ = None
        self.classes_ = None

    def fit(self, X: csr_matrix, y: np.ndarray):
        # Ensure X is in CSR format for efficient row slicing
        if not isinstance(X, csr_matrix):
            X = X.tocsr() # Convert to CSR if it's not already

        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        class_count = np.zeros(n_classes, dtype=np.float64)
        feature_count = np.zeros((n_classes, n_features), dtype=np.float64)

        # sum features per class
        for ci, c in enumerate(self.classes_):
            mask = (y == c)
            Xc = X[mask.nonzero()[0]]
            class_count[ci] = mask.sum()
            feature_count[ci] = np.asarray(Xc.sum(axis=0)).ravel()

        # log prior
        self.class_log_prior_ = np.log(class_count / class_count.sum())

        # log likelihood with Laplace smoothing
        smoothed_fc = feature_count + self.alpha
        smoothed_cc = smoothed_fc.sum(axis=1, keepdims=True)
        self.feature_log_prob_ = np.log(smoothed_fc / smoothed_cc)
        return self

    def predict_log_proba(self, X: csr_matrix):
        # log P(y) + sum x_i * log P(x_i|y)
        jll = X @ self.feature_log_prob_.T
        jll = jll + self.class_log_prior_
        # log-softmax
        amax = np.max(jll, axis=1, keepdims=True)
        lse = amax + np.log(np.sum(np.exp(jll - amax), axis=1, keepdims=True))
        return jll - lse

    def predict(self, X: csr_matrix):
        logp = self.predict_log_proba(X)
        idx = np.argmax(logp, axis=1)
        return self.classes_[idx]

nb = MultinomialNB_NoSklearn(alpha=1.0)
nb.fit(Xtr, y_train)
pred = nb.predict(Xte)

acc = (pred == y_test).mean()
print("Accuracy:", acc)


Accuracy: 0.7872340425531915


In [48]:
# Confusion matrix sederhana (tanpa sklearn)
def confusion_matrix_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(((y_true==1) & (y_pred==1)).sum())
    tn = int(((y_true==0) & (y_pred==0)).sum())
    fp = int(((y_true==0) & (y_pred==1)).sum())
    fn = int(((y_true==1) & (y_pred==0)).sum())
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp}

def metrics_from_cm(cm):
    tn, fp, fn, tp = cm["tn"], cm["fp"], cm["fn"], cm["tp"]

    total = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

cm = confusion_matrix_binary(y_test, pred)
m = metrics_from_cm(cm)

# bikin tabel rapi
metrics_table = pd.DataFrame([{
    "Accuracy": round(m["accuracy"], 4),
    "Precision": round(m["precision"], 4),
    "Recall": round(m["recall"], 4),
    "F1-Score": round(m["f1_score"], 4),
    "TP": cm["tp"],
    "TN": cm["tn"],
    "FP": cm["fp"],
    "FN": cm["fn"],
}])

metrics_table


,Accuracy,Precision,Recall,F1-Score,TP,TN,FP,FN
0,0.7872,0.8571,0.4,0.5455,6,31,1,9


## 8) Coba prediksi manual

In [50]:
def predict_text(text: str):
    t = basic_clean_competition(text)
    Xq = tfidf_matrix(np.array([t]), vocab, df_counts, n_docs_train=len(X_train))
    p = nb.predict(Xq)[0]
    return "positive" if int(p) == 1 else "negative"

samples = [
    "MBG ini kacau banget, banyak anak keracunan.",
    "Programnya ribet, makanannya telat terus, bikin kesel."
]
for s in samples:
    print(s, "->", predict_text(s))


MBG ini kacau banget, banyak anak keracunan. -> negative
Programnya ribet, makanannya telat terus, bikin kesel. -> negative
